## Reddit thread scraper with Frontier LLMs in Python

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# scraper utilities
from bs4 import BeautifulSoup
import requests


# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]


def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]


In [4]:
# set up environment

MODEL_LLAMA = 'llama3.2:1b'
OLLAMA_BASE_URL = "http://127.0.0.1:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [17]:
content = fetch_website_contents("https://www.reddit.com/r/mbti/comments/1pzp59z/is_chatgpt_informative_on_mbti_theory_mbti/")
content
links = fetch_website_links("https://www.reddit.com/r/mbti/comments/1pzp59z/is_chatgpt_informative_on_mbti_theory_mbti/")
links

['#main-content',
 '/',
 'https://www.reddit.com/login/',
 '/r/mbti/',
 '/r/mbti/',
 '/user/mamacorsica/',
 'https://www.reddit.com/r/mbti/comments/1pzp59z/is_chatgpt_informative_on_mbti_theory_mbti/?tl=pt-br',
 'https://www.reddit.com/r/mbti/comments/1pzp59z/is_chatgpt_informative_on_mbti_theory_mbti/?tl=th',
 'https://www.reddit.com/r/mbti/comments/1pzp59z/is_chatgpt_informative_on_mbti_theory_mbti/?tl=de',
 'https://www.redditinc.com/policies/user-agreement',
 'https://www.redditinc.com/policies/privacy-policy',
 'https://www.redditinc.com/policies/content-policy',
 'https://www.reddit.com/policies/privacy-policy',
 'https://www.redditinc.com/policies/user-agreement',
 'https://support.reddithelp.com/hc/sections/38303584022676-Accessibility',
 'https://redditinc.com']

In [14]:
# set up prompts

system_prompt = """
You are an assistant that analyzes the contents of a website,
and provides a short summary.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes comments, then summarize them too.

"""

In [15]:
def create_summary(url):
    stream = ollama.chat.completions.create(
        model= MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt + fetch_website_contents(url)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response),display_id=display_handle.display_id)
    

In [16]:
create_summary("https://www.reddit.com/r/mbti/comments/1pzp59z/is_chatgpt_informative_on_mbti_theory_mbti/")

This website appears to be showcasing Reddit as the "heart of the internet", promoting the site as a hub for discussion and community sharing.

The comments section is not present in this excerpt. However, based on previous answers provided by Reddit that mentions MBTI theory and psychology, it seems that they do provide information about the theory, including types (introvert vs extrovert), loops (e.g. INTP, ISFJ), and grips (e.g. INFJ, ENTJ).